# TKCE — Deep-dive: training dynamics (standalone)

Deep Siamese encoder + deep TabResNet on ONE balanced binary dataset
(`eye_movements`: 7,608 rows, 20 features, 50/50), long training, slow LR,
**no early stopping**, repeated over several random shuffles.

**Setup:** `Runtime → Change runtime type → A100 (or L4) GPU`.
Run cells top to bottom. Cell 5 (lambda sweep) is the one to run first.

## 1. Config

In [ ]:
REPO_URL   = "https://github.com/sushanedulloo/TKCE.git"
DRIVE_BASE = "/content/drive/MyDrive/tkce_results"

# ---- deep + long + SLOW ----
TASK      = 361070      # eye_movements: 7,608 rows / 20 features, balanced
SEEDS     = "0,1,2"     # shuffles (a new random split each)
EPOCHS    = 600
LR        = 1e-6        # slow: training was reaching AUC 1 too quickly
BATCH     = 64          # TASK batch  -> 84 steps/epoch + gradient noise
CBATCH    = 256         # CONTRASTIVE batch -> keeps 510 in-batch negatives
ENC_WIDTH = 512
ENC_DEPTH = 6
EMB       = 128
N_BLOCKS  = 8
D         = 256
D_HIDDEN  = 512
HEAD      = "tabresnet"

# contrastive term is a REGULARIZER -> the useful lambda range is small
LAMBDAS   = "0,0.005,0.015,0.05,0.15,0.5"
LOSSES    = "infonce,aninfonce,clip_infonce,contrastive,triplet"

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
FIG = DRIVE_BASE + "/figures"
AN  = DRIVE_BASE + "/analysis"
LOG = DRIVE_BASE + "/deep_joint_run.log"
for d in (DRIVE_BASE, FIG, AN): os.makedirs(d, exist_ok=True)
print('Saving to:', FIG, 'and', AN)

## 3. Get the code + install

In [ ]:
%cd /content
!rm -rf tkce_deep
!git clone $REPO_URL tkce_deep
%cd tkce_deep
!pip install -q -r requirements-tkce.txt
print('Ready')

## 4. GPU check

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
!nvidia-smi -L

## 5. LAMBDA SWEEP  ← run this first
Sweeps the fused-loss weight for one loss. Shows TEST AUC vs $\lambda$ and how
much of the fused loss the contrastive term takes at each $\lambda$.

In [ ]:
!python -u run_deep_joint.py --task $TASK --seeds $SEEDS --epochs $EPOCHS --lr $LR --batch-size $BATCH --contrastive-batch-size $CBATCH --enc-width $ENC_WIDTH --enc-depth $ENC_DEPTH --embedding-dim $EMB --d $D --d-hidden $D_HIDDEN --n-blocks $N_BLOCKS --head $HEAD --device auto --losses infonce --lambdas $LAMBDAS --tag lambdasweep --out "$FIG" --csv "$AN" 2>&1 | tee "$LOG"

## 6. LOSS COMPARISON at the best lambda
Read the best $\lambda$ off the sweep above, set it below, then compare all 5 losses.

In [ ]:
BEST_LAMBDA = "0.015"   # <-- set from the lambda sweep result
!python -u run_deep_joint.py --task $TASK --seeds $SEEDS --epochs $EPOCHS --lr $LR --batch-size $BATCH --contrastive-batch-size $CBATCH --enc-width $ENC_WIDTH --enc-depth $ENC_DEPTH --embedding-dim $EMB --d $D --d-hidden $D_HIDDEN --n-blocks $N_BLOCKS --head $HEAD --device auto --losses $LOSSES --lambdas $BEST_LAMBDA --tag losssweep --out "$FIG" --csv "$AN"

## 7. (Optional) Let the model LEARN the balance
Uncertainty weighting instead of a hand-set $\lambda$.

In [ ]:
!python -u run_deep_joint.py --task $TASK --seeds $SEEDS --epochs $EPOCHS --lr $LR --batch-size $BATCH --contrastive-batch-size $CBATCH --enc-width $ENC_WIDTH --enc-depth $ENC_DEPTH --embedding-dim $EMB --d $D --d-hidden $D_HIDDEN --n-blocks $N_BLOCKS --head $HEAD --device auto --losses infonce --lambdas 0.5 --uncertainty-weighting --tag learned --out "$FIG" --csv "$AN"

## 8. Show the figures

In [ ]:
from IPython.display import Image, display
import glob
for p in sorted(glob.glob(FIG + '/deep_joint_*.png')):
    print(p); display(Image(p))

## 9. Peek at the numbers

In [ ]:
import pandas as pd, glob
for t in sorted(glob.glob(AN + '/deep_joint_*_test.csv')):
    print('===', t); tdf = pd.read_csv(t)
    key = 'lam' if tdf.lam.nunique() > 1 else 'loss'
    display(tdf.groupby(key)[['test_auc','test_acc','train_auc_final']].agg(['mean','std']).round(4))

## Tips
- Curves almost flat? LR `1e-6` may be too slow — try `1e-5` or more epochs.
- `train_auc_final` near 1.0 = the model memorised the 5,325 training rows
  (it has ~3.5M parameters) — lower LR / smaller model / more regularisation.
- Each cell writes its own tagged files, so nothing overwrites.